In [17]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
import pinecone
from sentence_transformers import SentenceTransformer

In [2]:
files = pd.read_csv("course_descriptions.csv", encoding = "ANSI")

### **We'll merge the tabular data from each column in a row into strings and then upload it to the vector store**

In [3]:
def create_course_description(row):
    return f'''The course name is {row["course_name"]}, the slug is {row["course_slug"]},
            the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''

In [5]:
pd.set_option('display.max_rows', 106)
files['course_description_new'] = files.apply(create_course_description, axis = 1)
print(files["course_description_new"])

0      The course name is Introduction to Tableau, th...
1      The course name is The Complete Data Visualiza...
2      The course name is Introduction to R Programmi...
3      The course name is Data Preprocessing with Num...
4      The course name is Introduction to Data and Da...
5      The course name is Data Cleaning and Preproces...
6      The course name is Introduction to Business An...
7      The course name is Data Analysis with Excel Pi...
8      The course name is SQL, the slug is sql,\n    ...
9      The course name is Credit Risk Modeling in Pyt...
10     The course name is Python Programmer Bootcamp,...
11     The course name is SQL + Tableau + Python, the...
12     The course name is Introduction to Jupyter, th...
13     The course name is Statistics, the slug is sta...
14     The course name is Mathematics, the slug is ma...
15     The course name is Introduction to Excel, the ...
16     The course name is Probability, the slug is pr...
17     The course name is Start

In [7]:
%load_ext dotenv
%dotenv

In [8]:
load_dotenv(find_dotenv(), override = True)

True

In [10]:
pc = Pinecone(api_key = os.environ.get("PINECONE_API_KEY"))

In [13]:
index_name = "my-index"
dimension = 384
metric = "cosine"

In [14]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
    print(f"{index_name} not in index list.")

my-index succesfully deleted.


In [15]:
pc.create_index(
    name = index_name,
    dimension = dimension,
    metric = metric,
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
    )
)

Name:,my-index
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://my-index-e9us1v6.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [16]:
index = pc.Index(index_name)

### **Embedding the data**

In [18]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3197.28it/s]


In [19]:
def create_embeddings(row):
    combined_text = ' '.join([str(row[field]) for field in ['course_description', 'course_description_new', 'course_description_short']])
    embedding = model.encode(combined_text, show_progress_bar = False)
    return embedding

In [20]:
files["embedding"] = files.apply(create_embeddings, axis = 1)

In [21]:
vectors_to_upsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectors = vectors_to_upsert)

UpsertResponse(upserted_count=106)

### **Semantic search**

In [22]:
query = "clustering"
query_embedding = model.encode(query, show_progress_bar=False).tolist()

In [24]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12,
    include_values = True
)

In [25]:
query_results

QueryResponse(matches=[ScoredVector(id='Machine Learning in Excel', score=0.35443, values=[-0.0183002222, -0.0279485844, -0.0253203418, ...381 more]), ScoredVector(id='Machine Learning with K-Nearest Neighbors', score=0.313393563, values=[-0.102056794, -0.0829931349, 0.0194283687, ...381 more]), ScoredVector(id='Machine Learning in Python', score=0.282739311, values=[-0.0447243787, -0.0423180237, -0.00483980449, ...381 more]), ScoredVector(id='Customer Churn Analysis with SQL and Tableau', score=0.282348037, values=[-0.0255761575, -0.0562175289, -0.0636670664, ...381 more]), ScoredVector(id='Growth Analysis with SQL, Python, and Tableau  ', score=0.259579, values=[0.0178535841, -0.0293453243, -0.0790976211, ...381 more]), ScoredVector(id='Linear Algebra and Feature Selection', score=0.25878, values=[-0.0354893208, -0.0207994021, 0.0170970224, ...381 more]), ScoredVector(id='Customer Engagement Analysis with SQL and Tableau', score=0.235071421, values=[0.0382172, -0.0132274805, -0.05340

In [27]:
score_threshold = 0.3
for match in query_results["matches"]:
    if match['score'] >= score_threshold:
        print(f"Matched item ID: {match['id']}, score: {match['score']}")

Matched item ID: Machine Learning in Excel, score: 0.35443
Matched item ID: Machine Learning with K-Nearest Neighbors, score: 0.313393563
